## 1: Introdução

---

### 1.1. Contextualização

Durante nossas pesquisas no projeto 1, vimos que uma aplicação dos modelos de Naive Bayes é de análise de sentimentos em textos corridos. Por exemplo, um modelo de Naive Bayes pode usar da contagem de palavras em cada texto para determinar se um tweet tem conotação positiva ou negativa por parte do usuário, como foi feito na Indonésia durante o ínicio da pandemia COVID-19.

Com base nisso, o problema sobre o qual decidimos nos debruçar é o de classificar o sentimento de resenhas de filmes como positivo ou negativo.


### 1.2. Objetivo

Os nossos objetivos são testar o algoritmo nesse caso e 


### 1.3. Descrição dos Dados

O dataset que escolhemos para aplicar o modelo é chamado ["IMDB Dataset of 50K Movie Reviews"](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews), disponível no Kaggle. Ao que o criador do dataset indica, esse conjunto de dados é uma amostra de um conjunto de treino para processamento de linguagem natural da universidade de Stanford.

Como o nome afirma, é composto por 50 mil exemplos de resenhas de filmes desconhecidos, todos eles rotulados com uma emoção, positivo ou negativo. Cada amostra é composta por apenas esses dois valores, rotulados como `review` e `sentiment` respectivamente. Vamos explorar os dados agora.

## 2: Análise Exploratória de Dados

---

### 2.1. Carregamento e Limpeza dos Dados

Vamos começar carregando o dataset (e algumas livrarias que vamos usar ao longo do projeto):

In [ ]:
install.packages("tm")
library(tm)

dataset <- read.csv("data/IMDB Dataset.csv")

head(dataset, n = 2)

,review,sentiment
,<chr>,<chr>
1,"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences, forget charm, forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal, I couldn't say I was ready for it, but as I watched more, I developed a taste for Oz, and got accustomed to the high levels of graphic violence. Not just violence, but injustice (crooked guards who'll be sold out for a nickel, inmates who'll kill on order and get away with it, well mannered, middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz, you may become comfortable with what is uncomfortable viewing....thats if you can get in touch with your darker side.",positive
2,"A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only ""has got all the polari"" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master's of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional 'dream' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell's murals decorating every surface) are terribly well done.",positive


Notamos de cara que o texto precisa ser tratado extensivamente, mas deixaremos isso para mais tarde.

Agora vamos verificar a situação geral dos dados usando a função `summary` para nos informar quantos elementos há no total, quantas duplicatas possuímos e quantos dados faltantes temos:

In [2]:
summary(dataset)

       review          sentiment    
 Length   :50000   Length   :50000  
 N.unique :49582   N.unique :    2  
 N.blank  :    0   N.blank  :    0  
 Min.nchar:   32   Min.nchar:    8  
 Max.nchar:13704   Max.nchar:    8  

Vemos que **não existem dados faltantes** felizmente, uma vez que `N.blank = 0`.

No entanto, podemos ver que nem todas as resenhas são únicas; em especial, apenas `49582` são únicas, restando `418` duplicatas. Para prevenir o enviesamento do modelo de Bayes (que é sensível a duplicatas), decidimos remover esses valores, uma vez que o conjunto é grande o suficiente para que uma perda pequena de dados não traga consequências significativas ao desempenho do algoritmo.

In [3]:
dataset <- dataset[!duplicated(dataset), ]

summary(dataset)

       review          sentiment    
 Length   :49582   Length   :49582  
 N.unique :49582   N.unique :    2  
 N.blank  :    0   N.blank  :    0  
 Min.nchar:   32   Min.nchar:    8  
 Max.nchar:13704   Max.nchar:    8  

### 2.2. Análise Univariada

...

### 2.3. Análise Bivariada e Multivariada

...

### 2.4. Insights

...

## 3: Validação Estatística

---

### 3.1. Definição das Hipóteses

...

### 3.2. Escolha a Execução do Teste

...

### 3.3. Interpretação do p-valor

...

### 2.2. Análise Univariada

...

## 4: Pré-processamento e Aplicação da Técnica

---

### 4.1. Preparação dos Dados para o Modelo:

Como vimos anteriormente, o texto precisa ser tratado para garantir a qualidade do significado das palavras a serem usadas na contagem para o algoritmo do Naive Bayes. 

Em outras palavras, precisamos garantir que as palavras com as quais o modelo vai treinar têm um "motivo" para serem contabilizadas na contagem. Por exemplo, artigos e preposições como `the`, `on`, `at`, `in` não adicionam muito ao significado do texto, mas sim servem para conectar ideias relevantes. Esse tipo de palavra é chamado de **stop word**. 

Para além das stop words, vemos que alguns elementos HTML (notavelmente o `br`) estão presentes em algumas resenhas. Esses elementos não tem significado nenhum pois serviram apenas para formatação. Também vale a pena remover números no texto, que isoladamente não servem muito.

Ademais, a palavra `"bad"` tem o mesmo significado que `"bad,"` ou `"bad..."`, e a pontuação por si só não enriquece muito o texto para fins de classificação de sentimento. No entanto, note que aqui temos uma exceção que queremos testar na forma do ponto de exclamação `!` (vide projeto 1) e, ainda nessa linha, queremos manter palavras que estão totalmente em maiúsculo e não são stop words.

Portanto, **para treinar o modelo**, iremos definir algumas funções que tratam o texto de uma resenha da seguinte forma:
- Removendo todas as stop words;
- pondo em minúsculo tudo que não estiver TOTALMENTE EM MAIÚSCULO;
- removendo toda a pontuação presente no texto exceto pelo ponto de exclamação;
- removendo os elementos HTML; e
- removendo os números no texto.

Essa transformação não será perfeita, pois não temos como prever erros de digitação dos resenhistas, por exemplo, mas tornará o treinamento do modelo mais enxuto.

In [ ]:
# função auxiliar para criar mais stop words
totitle <- function(word) {
  paste(
    toupper(substring(word, 1, 1)),
    tolower(substring(word, 2)),
    sep = ""
  )
}

case_insensitive_stopwords <- function() {
  lowercase <- tm::stopwords("SMART") # i, me, myself...
  titlecase <- totitle(lowercase)     # I, Me, Myself...
  uppercase <- toupper(lowercase)     # I, ME, MYSELF...

  c(rbind(lowercase, titlecase, uppercase))
}

clean_reviews <- function(reviews) {
  stop_words <- case_insensitive_stopwords()

  reviews <- tm::removeNumbers(reviews)
  reviews <- tm::removeWords(reviews, stop_words)

  html <- "<\\w* \\/>" # <br /> e similares ...
  reviews <- gsub(html, " ", reviews, perl = TRUE)

  # removendo pontuação MAS mantendo o "!"
  # o regex é: um ou mais " # $ % & ' etc.
  punctuation_no_excl <- "[\"#$%&'\\()*+,\\-./:;<=>?@[\\]^_`{|}~]+"
  reviews <- gsub(punctuation_no_excl, " ", reviews, perl = TRUE)

  # SE a palavra do começo ao fim for composta por
  # 1 ou mais letras maiúsculas, ignore (?![A-Z]+\W)
  # Caso contrário, selecione tudo (\S*)
  not_uppercase_word <- "(?![A-Z]+\\W)(\\S*)"
  reviews <- gsub(not_uppercase_word, "\\L\\1", reviews, perl = TRUE)

  # separando os pontos de exclamação para poder
  # contar eles como uma "palavra" separada
  reviews <- gsub("!", " ! ", reviews)

  reviews
}

Vamos testar essa função em uma resenha qualquer:

In [7]:
random_number <- sample(nrow(dataset), 1)

review <- dataset$review[random_number]
review

clean <- clean_reviews(review)
clean

[1] "Using footage pillaged from Planet of Dinosaurs this shot on video (except for the stolen footage) concerns a bunch of people shot into space who land on a dinosaur planet that is...don't wait for it, is really earth. Its a five minute sketch stretched to 90 minutes. Slightly better than Chickboxer (another in the Bad Movie Police series)-having a nostalgic home movie feel coupled with good stolen effects, this movie is still an impossible slog to get through. I'm left to ponder the question are we becoming so uncreative that we're now pillaging old movies not only for plot but also for mismatched footage? Clearly low budget producers are getting so desperate they really will give us anything to take our money"

[1] " footage pillaged  planet  dinosaurs  shot  video     stolen footage  concerns  bunch  people shot  space  land   dinosaur planet    wait      earth     minute sketch stretched   minutes  slightly   chickboxer     bad movie police series   nostalgic home movie feel coupled  good stolen effects   movie    impossible slog      left  ponder  question     uncreative    pillaging  movies    plot    mismatched footage   low budget producers    desperate    give      money"

Vemos que mantemos em boa parte as palavras com mais relevância para nossa tarefa, então podemos prosseguir.

aqui: falta criar mais atributos (n de palavras e tal)

### 4.2. Modelagem

...

## 5: Resultados e Interpretação

---

### 5.1. Apresentação dos Resultados

...

### 5.2. Análise Crítica dos Resultados

...

## 6: Conclusão
---

### 6.1. Resumo dos Insights

...

### 6.2. Recomendações Práticas

...

### 6.3. Limitações e Próximos Passos

...